In [10]:
from dataclasses import dataclass, field
from typing import List, Optional


# ============================================================
# HOW TO UPDATE THIS FILE
# ------------------------------------------------------------
# 1. When you get a mark back, change score=None to the mark.
# 2. If the mark is not out of 100, also set max_score properly.
# 3. If you want to predict a future mark manually, fill in predicted=...
# 4. If predicted is left as None, the script will estimate using:
#       - your current completed average in that module
#       - otherwise a default fallback target
# ============================================================


DEFAULT_FALLBACK_TARGET = 71.0  # change this if you want a different default prediction


@dataclass
class Assessment:
    name: str
    weight: float                    # weight inside module, in %
    score: Optional[float] = None    # raw mark received
    max_score: float = 100.0         # maximum possible raw mark
    predicted: Optional[float] = None  # optional manual prediction out of 100

    def score_as_percent(self) -> Optional[float]:
        if self.score is None:
            return None
        return 100.0 * self.score / self.max_score


@dataclass
class Module:
    name: str
    credits: int
    assessments: List[Assessment] = field(default_factory=list)

    def completed_weight(self) -> float:
        return sum(a.weight for a in self.assessments if a.score is not None)

    def current_mark_completed_only(self) -> Optional[float]:
        """
        Average on returned work only.
        """
        completed_w = self.completed_weight()
        if completed_w == 0:
            return None
        total = sum(
            a.weight * a.score_as_percent() / 100.0
            for a in self.assessments
            if a.score is not None
        )
        return 100.0 * total / completed_w

    def banked_mark(self) -> float:
        """
        Mark currently banked in the module, counting missing work as zero for now.
        """
        total = 0.0
        for a in self.assessments:
            pct = a.score_as_percent()
            if pct is not None:
                total += a.weight * pct / 100.0
        return total

    def projected_mark(self, fallback_target: float = DEFAULT_FALLBACK_TARGET) -> float:
        """
        Projected full module mark.
        Priority for missing marks:
          1. manual predicted value
          2. current average on completed work in this module
          3. fallback_target
        """
        current_avg = self.current_mark_completed_only()

        total = 0.0
        for a in self.assessments:
            pct = a.score_as_percent()
            if pct is not None:
                use_pct = pct
            elif a.predicted is not None:
                use_pct = a.predicted
            elif current_avg is not None:
                use_pct = current_avg
            else:
                use_pct = fallback_target

            total += a.weight * use_pct / 100.0

        return total


def weighted_degree_average(modules: List[Module], mark_getter) -> float:
    total_credits = sum(m.credits for m in modules)
    weighted_sum = sum(mark_getter(m) * m.credits for m in modules)
    return weighted_sum / total_credits


def print_summary(modules: List[Module], fallback_target: float = DEFAULT_FALLBACK_TARGET) -> None:
    print("=" * 72)
    print("MODULE SUMMARY")
    print("=" * 72)

    for m in modules:
        completed = m.completed_weight()
        current_completed = m.current_mark_completed_only()
        banked = m.banked_mark()
        projected = m.projected_mark(fallback_target)

        print(f"\n{m.name} ({m.credits} credits)")
        print("-" * 72)

        for a in m.assessments:
            score_text = (
                f"{a.score}/{a.max_score} ({a.score_as_percent():.1f}%)"
                if a.score is not None
                else "Not yet returned"
            )
            pred_text = f", predicted={a.predicted:.1f}%" if a.predicted is not None else ""
            print(f"  - {a.name:<30} weight={a.weight:>5.1f}%   score={score_text}{pred_text}")

        print(f"  Completed weight:              {completed:.1f}%")
        print(
            f"  Current average (completed):   "
            f"{current_completed:.2f}%"
            if current_completed is not None
            else "  Current average (completed):   N/A"
        )
        print(f"  Banked mark so far:            {banked:.2f}%")
        print(f"  Projected final module mark:   {projected:.2f}%")

    print("\n" + "=" * 72)
    print("DEGREE-LEVEL TOTALS")
    print("=" * 72)

    # Completed-only degree average:
    # weighted by credits * fraction of each module already completed
    numerator = 0.0
    denominator = 0.0
    for m in modules:
        current_completed = m.current_mark_completed_only()
        completed_fraction = m.completed_weight() / 100.0
        if current_completed is not None and completed_fraction > 0:
            numerator += m.credits * completed_fraction * current_completed
            denominator += m.credits * completed_fraction

    completed_only_degree_avg = numerator / denominator if denominator > 0 else None

    banked_degree_avg = weighted_degree_average(modules, lambda m: m.banked_mark())
    projected_degree_avg = weighted_degree_average(
        modules, lambda m: m.projected_mark(fallback_target)
    )

    if completed_only_degree_avg is not None:
        print(f"Current average on completed work only: {completed_only_degree_avg:.2f}%")
    else:
        print("Current average on completed work only: N/A")

    print(f"Banked average so far:                  {banked_degree_avg:.2f}%")
    print(f"On-track average:                       {projected_degree_avg:.2f}%")
    print("=" * 72)


# ============================================================
# YOUR MODULE DATA
# ============================================================

modules = [
    Module(
        name="Dissertation in Particle & Nuclear Physics",
        credits=60,
        assessments=[
            Assessment("Dissertation", weight=70.0, score=None),
            Assessment("Presentation", weight=10.0, score=None),
            Assessment("Performance", weight=20.0, score=None),
        ],
    ),

    Module(
        name="Research Skills in Particle & Nuclear Physics",
        credits=20,
        assessments=[
            Assessment("Presentation", weight=20.0, score=56),
            Assessment("Poster Presentation", weight=10.0, score=66),
            Assessment("Group Project", weight=20.0, score=None, predicted = 73),            # 66.6% of 30%
            Assessment("Project Individual Essay", weight=10.0, score=None, predicted = 60), # 33.3% of 30%
            Assessment("Dissertation Proposal Report", weight=20.0, score=None, predicted = 68),
            Assessment("Dissertation Proposal Oral", weight=20.0, score=None, predicted = 68),
        ],
    ),

    Module(
        name="Detectors in Particle & Nuclear Physics",
        credits=10,
        assessments=[
            Assessment("Pres 1", weight=15.0, score=54),   # 50% of 30%
            Assessment("Pres 2", weight=15.0, score=70),   # 50% of 30%
            Assessment("CP1", weight=10.0, score=65),      # 20% of 50%
            Assessment("CP2", weight=10.0, score=75),      # 20% of 50%
            Assessment("CP3", weight=15.0, score=65),      # 30% of 50%
            Assessment("CP4", weight=15.0, score=35),      # 30% of 50%
            Assessment("Outreach Presentation", weight=20.0, score=73),
        ],
    ),

    Module(
        name="Data Analysis and Machine Learning",
        credits=20,
        assessments=[
            Assessment("Report 1", weight=10.25, score=100),         # 12.5% of 82%
            Assessment("Report 2", weight=10.25, score=65),
            Assessment("Report 3", weight=10.25, score=None),
            Assessment("Report 4", weight=10.25, score=None),
            Assessment("Mini Viva Report 1", weight=10.25, score=75),
            Assessment("Mini Viva Report 2", weight=10.25, score=70),
            Assessment("Mini Viva Report 3", weight=10.25, score=None),
            Assessment("Mini Viva Report 4", weight=10.25, score=None),
            Assessment("Checkpoints", weight=18.0, score=None, max_score=18),
        ],
    ),

    Module(
        name="Lagrangian Dynamics",
        credits=10,
        assessments=[
            Assessment("Exam", weight=100.0, score=None, predicted = 65),
        ],
    ),

    Module(
        name="Quantum Computing Project",
        credits=10,
        assessments=[
            Assessment("Presentation", weight=25.0, score=None, predicted = 70),
            Assessment("Written Group Report", weight=60.0, score=None, predicted = 70),
            Assessment("Peer Mark", weight=15.0, score=None, predicted = 70),
        ],
    ),

    Module(
        name="General Relativity",
        credits=10,
        assessments=[
            Assessment("Exam", weight=100.0, score=None, predicted = 60),
        ],
    ),

    Module(
        name="Quantum Theory",
        credits=10,
        assessments=[
            Assessment("Exam", weight=100.0, score=None, predicted = 60),
        ],
    ),

    Module(
        name="Nuclear Physics",
        credits=10,
        assessments=[
            Assessment("Exam", weight=100.0, score=None, predicted = 75),
        ],
    ),

    Module(
        name="Particle Physics",
        credits=10,
        assessments=[
            Assessment("Exam", weight=100.0, score=82),
        ],
    ),

    Module(
        name="Current Topics in Particle Physics",
        credits=10,
        assessments=[
            Assessment("Oral Presentation", weight=50.0, score=None, predicted = 68),
            Assessment("Exam", weight=50.0, score=None, predicted = 60),
        ],
    ),
]


if __name__ == "__main__":
    print_summary(modules, fallback_target=DEFAULT_FALLBACK_TARGET)

MODULE SUMMARY

Dissertation in Particle & Nuclear Physics (60 credits)
------------------------------------------------------------------------
  - Dissertation                   weight= 70.0%   score=Not yet returned
  - Presentation                   weight= 10.0%   score=Not yet returned
  - Performance                    weight= 20.0%   score=Not yet returned
  Completed weight:              0.0%
  Current average (completed):   N/A
  Banked mark so far:            0.00%
  Projected final module mark:   71.00%

Research Skills in Particle & Nuclear Physics (20 credits)
------------------------------------------------------------------------
  - Presentation                   weight= 20.0%   score=56/100.0 (56.0%)
  - Poster Presentation            weight= 10.0%   score=66/100.0 (66.0%)
  - Group Project                  weight= 20.0%   score=Not yet returned, predicted=73.0%
  - Project Individual Essay       weight= 10.0%   score=Not yet returned, predicted=60.0%
  - Dissertation